# RL Jammer DSP acceleration prototype (`RL_Jammer_testing.ipynb`)

This notebook **implements** (not just documents) DSP-chain accelerations:
- vectorized/batched STFT feature extraction,
- cached DSP constants (window/frequency grid),
- decode scheduling for reward computation,
- batched decode calls through `rx_command_iq_broadcast`,
- resample short-circuit when sample rates match.


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Sequence, Tuple
import numpy as np
import torch
from pathlib import Path
import os
import advanced_link_skdsp_v7_robust as link7
import score_iq_decode as scorer


if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

from torch.utils.tensorboard import SummaryWriter

from accelerated_training_utils import (JammerVecEnv,
                                        repeat_to_length_mod,
                                        build_stft_observation_from_iq_batch,
                                        jammer_controller_batch)
from tx_controller_tone_pulse_stft_varlen_5 import (ActorCritic,
                                                    TonePulseTXControlNetVarLen,
                                build_controlled_tone_pulse_batch_from_iq_batches)

from accelerated_training_utils import (
    precompute_training_cache,
    create_cached_dataloader,

)



In [2]:



# Discrete action space size consumed by tx_controller_tone_pulse_stft_varlen_3.ActorCritic
ACTION_DIM = 75

In [3]:
_DSP_CACHE: Dict[Tuple[str, int, int, float], Dict[str, torch.Tensor]] = {}


def _get_stft_constants(device: torch.device,
                        nfft: int,
                        nperseg: int,
                        sample_rate_hz: float) -> Dict[str, torch.Tensor]:
    key = (str(device), int(nfft), int(nperseg), float(sample_rate_hz))
    cached = _DSP_CACHE.get(key)
    if cached is not None:
        return cached

    window = torch.hann_window(nperseg, dtype=torch.float32, device=device)
    fgrid = torch.fft.fftshift(
        torch.fft.fftfreq(nfft, d=1.0 / max(float(sample_rate_hz), 1.0)).to(device)
    ).to(torch.float32)
    out = {"window": window, "fgrid": fgrid}
    _DSP_CACHE[key] = out
    return out


def preprocess_batched_iq_to_stft_feature_fast(
    iq_batch: torch.Tensor,
    sample_rate_hz: float,
    nperseg: int = 128,
    noverlap: int = 96,
    nfft: int = 128,
    out_channels: int = 4,
) -> Dict[str, torch.Tensor]:
    """Vectorized STFT preprocessing for a complex IQ batch [B, T]."""
    if not torch.is_tensor(iq_batch):
        iq_batch = torch.as_tensor(iq_batch)
    if iq_batch.ndim != 2:
        raise ValueError(f"iq_batch must be [B, T], got {tuple(iq_batch.shape)}")

    x = iq_batch.to(dtype=torch.complex64)
    if x.shape[1] < 8:
        x = torch.nn.functional.pad(x, (0, 8 - x.shape[1]))

    x = x - x.mean(dim=1, keepdim=True)
    scale = torch.median(torch.abs(x), dim=1, keepdim=True).values
    x = x / (scale + 1e-6)

    nperseg_eff = int(min(nperseg, max(8, x.shape[1])))
    noverlap_eff = int(min(noverlap, nperseg_eff - 1))
    hop_length = max(1, nperseg_eff - noverlap_eff)

    const = _get_stft_constants(device=x.device, nfft=nfft, nperseg=nperseg_eff, sample_rate_hz=sample_rate_hz)

    z = torch.stft(
        x,
        n_fft=nfft,
        hop_length=hop_length,
        win_length=nperseg_eff,
        window=const["window"],
        center=False,
        return_complex=True,
    )
    z = torch.fft.fftshift(z, dim=1)

    mag = torch.log1p(torch.abs(z)).to(torch.float32)
    phase = torch.angle(z).to(torch.float32)
    real = z.real.to(torch.float32)
    imag = z.imag.to(torch.float32)
    power = (torch.abs(z) ** 2).to(torch.float32)
    power_log = torch.log1p(power)

    f = const["fgrid"][None, :, None]
    denom = power.sum(dim=1, keepdim=True) + 1e-12
    centroid = (f * power).sum(dim=1, keepdim=True) / denom
    centroid_plane = centroid.repeat(1, mag.shape[1], 1) / max(float(sample_rate_hz), 1.0)

    spread = torch.sqrt((((f - centroid) ** 2) * power).sum(dim=1, keepdim=True) / denom + 1e-12)
    spread_plane = (spread / max(float(sample_rate_hz), 1.0)).repeat(1, mag.shape[1], 1)

    flatness = torch.exp(torch.mean(torch.log(torch.clamp(power, min=1e-12)), dim=1, keepdim=True))
    flatness = flatness / (power.mean(dim=1, keepdim=True) + 1e-12)
    flatness_plane = flatness.repeat(1, mag.shape[1], 1)

    frame_power = power.mean(dim=1, keepdim=True)
    frame_power_norm = frame_power / (frame_power.mean(dim=2, keepdim=True) + 1e-12)
    frame_power_plane = frame_power_norm.repeat(1, mag.shape[1], 1)

    delta_t_mag = torch.diff(mag, dim=2, prepend=mag[:, :, :1])
    delta_f_mag = torch.diff(mag, dim=1, prepend=mag[:, :1, :])
    delta_t_power = torch.diff(power_log, dim=2, prepend=power_log[:, :, :1])
    delta_t_phase = torch.diff(phase, dim=2, prepend=phase[:, :, :1])

    sr_plane = torch.full_like(mag, torch.log10(torch.tensor(max(float(sample_rate_hz), 1.0), device=mag.device)) / 10.0)

    feat = torch.stack(
        [
            mag,
            phase,
            centroid_plane,
            sr_plane,
            real,
            imag,
            power_log,
            delta_t_mag,
            delta_f_mag,
            delta_t_phase,
            delta_t_power,
            spread_plane,
            flatness_plane,
            frame_power_plane,
        ][:out_channels],
        dim=1,
    )

    peak_bin = torch.argmax(power.mean(dim=2), dim=1)
    peak_hz = const["fgrid"][peak_bin]
    rx_power = (x.abs() ** 2).mean(dim=1)

    return {
        "feature": feat.to(torch.float32),
        "rx_power": rx_power.to(torch.float32),
        "peak_hz": peak_hz.to(torch.float32),
        "lengths": torch.full((x.shape[0],), float(x.shape[1]), device=x.device, dtype=torch.float32),
    }


In [4]:
def build_stft_observation_fast(
    *,
    iq1: torch.Tensor,
    iq2: torch.Tensor,
    iq3: torch.Tensor,
    intake_sample_rate_hz: float,
    device: str = "cpu",
) -> Dict[str, List[torch.Tensor]]:
    p1 = preprocess_batched_iq_to_stft_feature_fast(iq1, intake_sample_rate_hz)
    p2 = preprocess_batched_iq_to_stft_feature_fast(iq2, intake_sample_rate_hz)
    p3 = preprocess_batched_iq_to_stft_feature_fast(iq3, intake_sample_rate_hz)
    return {
        "stft_feature_list": [
            p1["feature"].to(device=device, dtype=torch.float32, non_blocking=True),
            p2["feature"].to(device=device, dtype=torch.float32, non_blocking=True),
            p3["feature"].to(device=device, dtype=torch.float32, non_blocking=True),
        ]
    }


In [5]:
@dataclass
class RewardScheduleConfig:
    decode_every_n_steps: int = 4
    proxy_reward_scale: float = 0.01


def _batched_decode(iq_batch: torch.Tensor, meta_list: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    if hasattr(link7, "rx_command_iq_broadcast"):
        return link7.rx_command_iq_broadcast(iq_batch, list(meta_list))
    return [link7.rx_command_iq(iq_batch[i], meta_list[i]) for i in range(iq_batch.shape[0])]


def compute_rewards_scheduled_batched(
    *,
    jam_batch: Sequence[Dict[str, Any]],
    samples: Sequence[Dict[str, Any]],
    jammer_sampling_freq: float,
    device: str,
    step_idx: int,
    sched: RewardScheduleConfig,
) -> Tuple[torch.Tensor, int, int]:
    """Batched reward computation with decode scheduling + resample short-circuit."""
    dev = torch.device(device)
    # do_decode = (step_idx % max(1, int(sched.decode_every_n_steps)) == 0)

    jammed_rows: List[torch.Tensor] = []
    metas: List[Dict[str, Any]] = []
    proxy_rewards: List[torch.Tensor] = []

    for jam_item, sample in zip(jam_batch, samples):
        tx_iq = jam_item["tx_iq"].to(dtype=torch.complex64)
        whole_iq = sample["whole_iq"].to(dtype=torch.complex64)
        whole_meta = sample["whole_meta"]
        sr_out = float(whole_meta["sample_rate_hz"])

        if abs(float(jammer_sampling_freq) - sr_out) < 1e-6:
            tx_rs = tx_iq
        else:
            tx_rs = link7.resample_iq(tx_iq, fs_in_hz=float(jammer_sampling_freq), fs_out_hz=sr_out)

        tx_rs = repeat_to_length_mod(tx_rs, int(whole_iq.shape[0]))
        tx_rs = torch.as_tensor(tx_rs[: whole_iq.shape[0]], dtype=torch.complex64)

        jammed = whole_iq.to(device) + tx_rs.to(device)
        jammed_rows.append(jammed.to(dev))
        metas.append(whole_meta)

        # proxy = -sched.proxy_reward_scale * torch.mean(torch.abs(tx_rs) ** 2)
        # proxy_rewards.append(proxy.to(torch.float32))

    if not jammed_rows:
        return torch.zeros(0, dtype=torch.float32, device=dev), 0, 0

    # if not do_decode:
    #     return torch.stack(proxy_rewards).to(device=dev), 0, 0

    max_len = max(int(x.numel()) for x in jammed_rows)
    jammed_pad = [torch.nn.functional.pad(x, (0, max_len - int(x.numel()))) for x in jammed_rows]
    jammed_batch = torch.stack(jammed_pad, dim=0).to(device=dev, dtype=torch.complex64)

    results = _batched_decode(jammed_batch, metas)

    scores: List[torch.Tensor] = []
    success = 0
    total = 0
    for r, m in zip(results, metas):
        total += 1
        s = float(scorer.score_decode(r, m)) if r is not None else 0.0
        if s > 0.0:
            success += 1
        scores.append(torch.tensor(s, dtype=torch.float32, device=dev))

    return torch.stack(scores), success, total


In [6]:
def _resolve_steps_per_epoch_for_notebook(env: JammerVecEnv, rollout_steps: int = 0) -> int:
    if int(rollout_steps) > 0:
        return int(rollout_steps)
    if hasattr(env, "_source_batches_per_epoch") and int(getattr(env, "_source_batches_per_epoch", 0)) > 0:
        return int(env._source_batches_per_epoch)
    sample_count = len(getattr(env, "samples", []))
    return max(1, int(np.ceil(sample_count / max(1, int(env.num_envs)))))


@torch.no_grad()
def _evaluate_split_metrics_batched_dsp(
    policy: torch.nn.Module,
    env: JammerVecEnv,
    split: str,
    device: str,
    value_coef: float,
    reward_sched: RewardScheduleConfig,
) -> Tuple[float, float]:
    split_samples = getattr(env, "test_samples", None) if split == "test" else getattr(env, "samples", None)
    if not split_samples:
        return float("inf"), 0.0

    logp_buf: List[torch.Tensor] = []
    rew_buf: List[torch.Tensor] = []
    val_buf: List[torch.Tensor] = []
    decode_successes = 0
    decode_total = 0
    eval_step_idx = 0

    original_mode = env.mode
    original_active = list(getattr(env, "_active", []))
    original_step_count = int(getattr(env, "_step_count", 0))

    try:
        env.set_mode(split)
        obs = env.reset()
        eval_steps = _resolve_steps_per_epoch_for_notebook(env, 0)

        for _ in range(eval_steps):
            model_obs = build_stft_observation_fast(
                iq1=obs["iq1"],
                iq2=obs["iq2"],
                iq3=obs["iq3"],
                intake_sample_rate_hz=float(env.jammer_sampling_freq),
                device=device,
            )
            action_t, value_t, logp_t = policy.get_action_value_logp(model_obs)
            actions = action_t.squeeze()
            if env.num_envs == 1 and isinstance(actions, torch.Tensor) and actions.ndim >= 1:
                action_batch = [actions]
            else:
                action_batch = actions

            if not getattr(env, "_active", None):
                env._active = env._next_samples()
            active_samples = list(env._active)

            jam_batch = jammer_controller_batch(
                model=env.model,
                samples=active_samples,
                actions=action_batch,
                jammer_sampling_freq=env.jammer_sampling_freq,
                user_peak_power_fraction=env.user_peak_power_fraction,
                device=env.device,
            )

            rewards_t, success, total = compute_rewards_scheduled_batched(
                jam_batch=jam_batch,
                samples=active_samples,
                jammer_sampling_freq=float(env.jammer_sampling_freq),
                device=device,
                step_idx=eval_step_idx,
                sched=reward_sched,
            )
            eval_step_idx += 1
            decode_successes += int(success)
            decode_total += int(total)

            env._step_count += 1
            done = env._step_count >= env.max_steps
            if done:
                env._active = env._next_samples()
                env._step_count = 0
            obs = env._obs_from_samples(env._active)

            logp_buf.append(logp_t)
            val_buf.append(value_t)
            rew_buf.append(rewards_t.to(device=device, dtype=torch.float32))

        returns = torch.stack(rew_buf, dim=0)
        values = torch.stack(val_buf, dim=0)
        logps = torch.stack(logp_buf, dim=0)
        if logps.ndim > returns.ndim:
            logps = logps.squeeze(-1)
        if values.ndim > returns.ndim:
            values = values.squeeze(-1)

        adv =  returns - values.detach()
        # adv = adv / (returns.std(unbiased=False) + 1e-8)
        policy_loss = -(logps * adv.detach()).mean()
        critic_loss = torch.nn.functional.mse_loss(values, returns)
        loss = policy_loss + value_coef * critic_loss
        decode_pct = (100.0 * decode_successes / decode_total) if decode_total else 0.0

        return float(loss.item()), float(decode_pct)
    finally:
        env.set_mode(original_mode)
        env._active = original_active
        env._step_count = original_step_count


def train_rl_loop_batched_dsp(
    policy: torch.nn.Module,
    env: JammerVecEnv,
    cfg: Any,
    reward_sched: RewardScheduleConfig,
):
    """PPO-style RL loop that uses one JammerVecEnv for train/test, epoch-wise eval, and checkpointing."""
    device = getattr(cfg, "device", "cuda" if torch.cuda.is_available() else "cpu")
    lr = float(getattr(cfg, "lr", 3e-4))
    epochs = max(1, int(getattr(cfg, "epochs", 1)))
    updates = max(1, int(getattr(cfg, "updates", 1)))
    value_coef = float(getattr(cfg, "value_coef", 0.5))
    rollout_steps = int(getattr(cfg, "rollout_steps", 0))

    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    steps_per_epoch = _resolve_steps_per_epoch_for_notebook(env, rollout_steps)

    writer = SummaryWriter(log_dir=getattr(cfg, "tensorboard_log_dir", "runs/train_rl_loop_batched_dsp"))
    checkpoint_dir = Path(getattr(cfg, "checkpoint_dir", "checkpoints_rl_batched_dsp"))
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_checkpoint_path = checkpoint_dir / getattr(cfg, "checkpoint_name", "best_model_batched_dsp.pt")
    best_test_loss = float("inf")

    ema_beta = 0.9
    ema_train_loss = None
    ema_test_loss = None

    env.set_mode("train")
    obs = env.reset()
    global_step = 0

    try:
        for epoch in range(epochs):
            env.set_mode("train")
            obs = env.reset()
            epoch_losses: List[float] = []
            epoch_success = 0
            epoch_total = 0

            for _ in range(updates):
                val_buf: List[torch.Tensor] = []
                logp_buf: List[torch.Tensor] = []
                rew_buf: List[torch.Tensor] = []

                for _ in range(steps_per_epoch):
                    model_obs = build_stft_observation_fast(
                        iq1=obs["iq1"],
                        iq2=obs["iq2"],
                        iq3=obs["iq3"],
                        intake_sample_rate_hz=float(env.jammer_sampling_freq),
                        device=device,
                    )
                    action_t, value_t, logp_t = policy.get_action_value_logp(model_obs)
                    actions = action_t.squeeze()

                    if env.num_envs == 1 and isinstance(actions, torch.Tensor) and actions.ndim >= 1:
                        action_batch = [actions]
                    else:
                        action_batch = actions

                    if not getattr(env, "_active", None):
                        env._active = env._next_samples()
                    active_samples = list(env._active)

                    jam_batch = jammer_controller_batch(
                        model=env.model,
                        samples=active_samples,
                        actions=action_batch,
                        jammer_sampling_freq=env.jammer_sampling_freq,
                        user_peak_power_fraction=env.user_peak_power_fraction,
                        device=env.device,
                    )

                    rewards_t, success, total = compute_rewards_scheduled_batched(
                        jam_batch=jam_batch,
                        samples=active_samples,
                        jammer_sampling_freq=float(env.jammer_sampling_freq),
                        device=device,
                        step_idx=global_step,
                        sched=reward_sched,
                    )
                    epoch_success += int(success)
                    epoch_total += int(total)

                    env._step_count += 1
                    done = env._step_count >= env.max_steps
                    if done:
                        env._active = env._next_samples()
                        env._step_count = 0
                    obs = env._obs_from_samples(env._active)

                    val_buf.append(value_t)
                    logp_buf.append(logp_t)
                    rew_buf.append(rewards_t.to(device=device, dtype=torch.float32))
                    global_step += 1

                returns = torch.stack(rew_buf, dim=0)
                values = torch.stack(val_buf, dim=0)
                logps = torch.stack(logp_buf, dim=0)
                if logps.ndim > returns.ndim:
                    logps = logps.squeeze(-1)
                if values.ndim > returns.ndim:
                    values = values.squeeze(-1)

                adv = returns - values.detach()
                # adv = adv / (returns.std(unbiased=False) + 1e-8)
                policy_loss = -(logps * adv.detach()).mean()
                critic_loss = torch.nn.functional.mse_loss(values, returns)
                loss = policy_loss + value_coef * critic_loss

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

                epoch_losses.append(float(loss.item()))

            train_loss_epoch = float(np.mean(epoch_losses)) if epoch_losses else 0.0
            train_decode_pct = (100.0 * epoch_success / epoch_total) if epoch_total else 0.0
            test_loss, test_decode_pct = _evaluate_split_metrics_batched_dsp(
                policy=policy,
                env=env,
                split="test",
                device=device,
                value_coef=value_coef,
                reward_sched=reward_sched,
            )

            ema_train_loss = train_loss_epoch if ema_train_loss is None else (ema_beta * ema_train_loss + (1.0 - ema_beta) * train_loss_epoch)
            ema_test_loss = test_loss if ema_test_loss is None else (ema_beta * ema_test_loss + (1.0 - ema_beta) * test_loss)

            writer.add_scalar("train/loss_epoch", train_loss_epoch, epoch)
            writer.add_scalar("train/loss_ema", ema_train_loss, epoch)
            writer.add_scalar("test/loss_epoch", test_loss, epoch)
            writer.add_scalar("test/loss_ema", ema_test_loss, epoch)
            writer.add_scalar("train/decode_success_pct", train_decode_pct, epoch)
            writer.add_scalar("test/decode_success_pct", test_decode_pct, epoch)

            if test_loss < best_test_loss:
                best_test_loss = test_loss
                torch.save(
                    {
                        "epoch": epoch,
                        "model_state_dict": policy.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "train_loss_epoch": train_loss_epoch,
                        "test_loss_epoch": test_loss,
                        "train_decode_success_pct": train_decode_pct,
                        "test_decode_success_pct": test_decode_pct,
                        "global_step": global_step,
                        "reward_schedule": {
                            "decode_every_n_steps": int(reward_sched.decode_every_n_steps),
                            "proxy_reward_scale": float(reward_sched.proxy_reward_scale),
                        },
                    },
                    best_checkpoint_path,
                )

            print(
                f"epoch={epoch + 1}/{epochs} "
                f"steps_per_epoch={steps_per_epoch} "
                f"train_loss={train_loss_epoch:.4f} "
                f"test_loss={test_loss:.4f} "
                f"train_decode_success_pct={train_decode_pct:.2f} "
                f"test_decode_success_pct={test_decode_pct:.2f}"
            )
    finally:
        writer.close()

    return policy


# Example:
# policy = train_rl_loop_batched_dsp(policy, env, cfg, RewardScheduleConfig(decode_every_n_steps=4))


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# epochs = 250
batch_size = 250
jammer_sampling_freq = 2e9

train_path_dat = Path("C:/Users/theon/Jamming/train_set_00/dataset")
test_path_dat = Path("C:/Users/theon/Jamming/test_set_00/dataset")
checkpoint_dir = Path("checkpoints_accelerated")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# 1) One-time deterministic precompute/cache of whole_iq + resampled iq1/iq2/iq3.
train_cache_dir = checkpoint_dir / "train_cache"
test_cache_dir = checkpoint_dir / "test_cache"
precompute_training_cache(train_path_dat, train_cache_dir, jammer_sampling_freq, section_len=1024, overwrite=False)
precompute_training_cache(test_path_dat, test_cache_dir, jammer_sampling_freq, section_len=1024, overwrite=False)

# 2) DataLoader path with worker prefetch + pinned memory.
num_workers = 0 if os.name == "nt" else 4
pin_memory = (device == "cuda")


train_loader = create_cached_dataloader(
    train_cache_dir,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    prefetch_factor=2,
    persistent_workers=(num_workers > 0),
)
test_loader = create_cached_dataloader(
    test_cache_dir,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    prefetch_factor=2,
    persistent_workers=(num_workers > 0),
)



In [8]:
@dataclass
class PPOConfig:
    rollout_steps: int = 0 #28
    updates: int = 50#1#00
    epochs: int = 200
    gamma: float = 0.99
    lr: float = 5e-4
    value_coef: float = 0.5
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    tensorboard_log_dir: str = "runs/train_rl_loop_18"
    checkpoint_dir: str = "checkpoints_rl_18"
    checkpoint_name: str = "best_model_18.pt"

In [9]:

max_tones = 4
action_dim = 9 + 3*max_tones

ac = ActorCritic(
        action_dim = action_dim,
        # in_ch = 14,
        base_ch = 24,
        max_tones = max_tones,
    ).to("cuda" if torch.cuda.is_available() else "cpu")

jve = JammerVecEnv(
        samples = train_loader,
        test_samples = test_loader,
        model = ac,
        jammer_sampling_freq = float(jammer_sampling_freq),
        num_envs = 1,
        reward_fn = None,
        user_peak_power_fraction= 1.0,
        max_steps = 1,
        device = device)

cfg = PPOConfig()
cfg.rollout_steps = jve._source_batches_per_epoch
test_no = 22

cfg.tensorboard_log_dir = f"runs/train_rl_loop_{test_no}"
cfg.checkpoint_dir = f"checkpoints_rl_{test_no}"
cfg.checkpoint_name = f"best_model_{test_no}.pt"
cfg.value_coef = 0.01
cfg.epochs = 200

rsc = RewardScheduleConfig()

rsc.decode_every_n_steps = 1

p = train_rl_loop_batched_dsp(
    policy = jve.model,
    env = jve,
    cfg = cfg,
    reward_sched = rsc)


p


RuntimeError: Given groups=1, weight of size [24, 14, 5, 5], expected input[1, 4, 128, 29] to have 14 channels, but got 4 channels instead